# 06 · Task 1 challengers — fine-tuned BERT and LLM few-shot
**Owner:** C (Task-1 lead) · **Phase 2** · Colab → Runtime → Change runtime type → **T4 GPU**

Same splits, same metrics module as notebook 03 → the three models are directly comparable.
LLM few-shot is scored on a stratified subset (20 per intent) of the test split to control API cost;
report that subset size, and score the other two models on the SAME subset too (done below).

In [1]:
import sys, os, subprocess
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:  # Colab: clone the team repo once per session, install requirements, load API keys from Secrets
    REPO_URL = "https://github.com/sh1or1tama1/banking-support-agent.git"
    if not os.path.exists("/content/banking-support-agent"):
        subprocess.run(["git", "clone", REPO_URL, "/content/banking-support-agent"], check=True)
    os.chdir("/content/banking-support-agent")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "-r", "requirements.txt"], check=True)
    from google.colab import userdata
    for k in ("OPENAI_API_KEY", "ANTHROPIC_API_KEY"):
        try: os.environ[k] = userdata.get(k)
        except Exception: pass
# os.environ["LLM_PROVIDER"] = "openai"; os.environ["LLM_MODEL"] = "<same model as Task 2>"
sys.path.insert(0, os.path.abspath(".." if os.path.basename(os.getcwd()) == "notebooks" else "."))

import joblib, pandas as pd
from IPython.display import display
from src import config as C
from src.utils import set_seed, sample_per_group
from src.data.load import load_banking77
from src.data.eval_sets import banking77_mapped
from src.classify import metrics as M
set_seed()

df = pd.read_parquet(C.SPLITS_FILE)
train, val, test = (df[df["split"] == s].copy() for s in ("train", "val", "test"))
b77 = banking77_mapped(load_banking77())
test_small = sample_per_group(test, "intent", 20)            # shared subset for all 3 models
print(len(train), len(val), len(test), "| subset", len(test_small), "| BANKING77 mapped", len(b77))

17882 3832 3831 | subset 520 | BANKING77 mapped 520


## 1. Fine-tune BERT (GPU)

In [2]:
from src.classify.bert import train_bert, BertIntent
BERT_DIR = C.OUTPUTS / "bert_intent"
if (BERT_DIR / "config.json").exists():   # trained elsewhere (e.g. notebooks/colab_train_bert.ipynb) -> reuse it
    history = pd.read_csv(BERT_DIR / "training_history.csv")
else:
    model, tok, labels, history = train_bert(train, val)
    history.to_csv(BERT_DIR / "training_history.csv", index=False)
history.to_csv(C.TABLES / "task1_bert_training_history.csv", index=False)
display(history[["epoch", "eval_accuracy", "eval_macro_f1", "eval_loss"]])
bert = BertIntent()

,epoch,eval_accuracy,eval_macro_f1,eval_loss
0,1.0,0.996608,0.996550,0.039000
1,2.0,0.998434,0.998361,0.011936
2,3.0,0.999478,0.999482,0.005025
3,4.0,0.999739,0.999741,0.003926


/Users/fangziming/Desktop/Uchicago课程/Multimodal/banking-support-agent/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


W1123 02:08:24.432000 64682 .venv/lib/python3.11/site-packages/torch/distributed/elastic/multiprocessing/redirects.py:35] NOTE: Redirects are currently not supported in MacOs.


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights: 100%|██████████| 201/201 [00:00<00:00, 6529.59it/s]

## 2. LLM few-shot (2 examples per intent from TRAIN)

In [3]:
from src.llm.client import LLMClient
from src.classify.llm_fewshot import LLMFewShotClassifier
fewshot = LLMFewShotClassifier(LLMClient(), train, k_per_intent=2)
print(fewshot.system[:1200])

You classify customer messages for a retail bank into exactly one intent.
Allowed intents (category: intent):
- CARD: activate_card
- CARD: activate_card_international_usage
- LOAN: apply_for_loan
- LOAN: apply_for_mortgage
- CARD: block_card
- CARD: cancel_card
- LOAN: cancel_loan
- LOAN: cancel_mortgage
- TRANSFER: cancel_transfer
- CARD: check_card_annual_fee
- CARD: check_current_balance_on_card
- FEES: check_fees
- LOAN: check_loan_payments
- LOAN: check_mortgage_payments
- ACCOUNT: check_recent_transactions
- ACCOUNT: close_account
- ACCOUNT: create_account
- CONTACT: customer_service
- ATM: dispute_ATM_withdrawal
- FIND: find_ATM
- FIND: find_branch
- PASSWORD: get_password
- CONTACT: human_agent
- TRANSFER: make_transfer
- ATM: recover_swallowed_card
- PASSWORD: set_up_password

Examples:
"I would like to activate a Master Card onmobile, help me" -> activate_card
"id lije to activate an amex where do i do it" -> activate_card
"I'm travelling overseas, how could I activate a cre

## 3. Compare all three on identical data

In [4]:
tfidf = joblib.load(C.OUTPUTS / "tfidf_lr.joblib")
from src.classify.tfidf_lr import predict_timed
rows, preds = [], {}
for set_name, d, text_col, gold_col in (("bitext_test_full", test, "instruction", "intent"),
                                        ("bitext_test_subset", test_small, "instruction", "intent"),
                                        ("banking77_mapped", b77, "text", "gold_intent")):
    texts = d[text_col].tolist()
    p, ms = predict_timed(tfidf, texts); rows.append(M.summary(d[gold_col], p, "tfidf_lr", set_name, ms)); preds[("tfidf_lr", set_name)] = p
    p, ms = bert.predict(texts);         rows.append(M.summary(d[gold_col], p, "bert_finetuned", set_name, ms)); preds[("bert_finetuned", set_name)] = p
    if set_name != "bitext_test_full":   # LLM only on the subset + external set (cost)
        r = fewshot.predict(texts)
        rows.append({**M.summary(d[gold_col], r["pred"], "llm_fewshot", set_name, 1000 * r["latency_s"].mean()),
                     "invalid_label_rate": round((r["pred"] == "INVALID").mean(), 4), "cost_usd": round(r["cost_usd"].sum(), 4)})
        preds[("llm_fewshot", set_name)] = r["pred"].to_numpy()
res = pd.DataFrame(rows); display(res)
res.to_csv(C.TABLES / "task1_all_models.csv", index=False)

llm-fewshot:   0%|          | 0/520 [00:00<?, ?it/s]

llm-fewshot: 100%|██████████| 520/520 [00:00<00:00, 5833.98it/s]

llm-fewshot:   0%|          | 0/520 [00:00<?, ?it/s]

llm-fewshot:  87%|████████▋ | 452/520 [00:00<00:00, 4518.41it/s]

llm-fewshot: 100%|██████████| 520/520 [00:00<00:00, 4645.13it/s]

,model,eval_set,n,accuracy,macro_f1,ms_per_query,invalid_label_rate,cost_usd
0,tfidf_lr,bitext_test_full,3831,0.9992,0.9991,0.040,NaN,NaN
1,bert_finetuned,bitext_test_full,3831,0.9992,0.9992,5.739,NaN,NaN
2,tfidf_lr,bitext_test_subset,520,0.9981,0.9981,0.049,NaN,NaN
3,bert_finetuned,bitext_test_subset,520,1.0000,1.0000,5.661,NaN,NaN
4,llm_fewshot,bitext_test_subset,520,0.9442,0.9429,589.744,0.0000,0.1371
5,tfidf_lr,banking77_mapped,520,0.4096,0.5158,0.048,NaN,NaN
6,bert_finetuned,banking77_mapped,520,0.4769,0.5674,13.942,NaN,NaN
7,llm_fewshot,banking77_mapped,520,0.6923,0.8531,613.313,0.2077,0.1367


In [5]:
# Robustness by Bitext flag (typos Z, colloquial Q, ...) for BERT vs TF-IDF on the full test split
test["pred"] = preds[("bert_finetuned", "bitext_test_full")]
display(M.accuracy_by_flag(test).rename(columns={"accuracy": "bert_acc"}))
M.plot_confusion(b77["gold_intent"], preds[("bert_finetuned", "banking77_mapped")], "task1_bert_confusion_banking77", "BERT · BANKING77 (real customers)")

,flag,n,bert_acc
0,B,3831,0.9992
1,C,3063,0.9997
2,E,391,1.0000
3,I,2694,1.0000
4,L,1920,1.0000
5,M,249,1.0000
6,P,1600,1.0000
7,Q,1924,0.9984
8,W,551,1.0000
9,Z,1960,0.9985


array([[0.925  , 0.     , 0.     , 0.05   , 0.     , 0.     , 0.     ,
        0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.025  ,
        0.     ],
       [0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     ],
       [0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
        0.     ],
       [0.     , 0.     , 0.     , 0.7    , 0.0375 , 0.     , 0.     ,
        0.     , 0.0125 , 0.0125 , 0.     , 0.     , 0.0625 , 0.05   ,
        0.     , 0.     , 0.0875 , 0.     , 0.     , 0.0125 , 0.025  ,
        0.     ],
       [0.     , 0.     , 0.     , 0.     , 0.     , 0.     , 0.     ,
     

## 4. Interpretation checklist for the report
- Bitext test: are all models near the ceiling? (expected - synthetic, templated data)
- BANKING77: how much does each model drop on real wording? Which handles it best, and why?
- Cost / latency per query: TF-IDF (ms, free) vs BERT (GPU ms) vs LLM (≈ seconds, $).
- 10-20 concrete errors: mapping ambiguity vs genuine model mistakes.
- Pick the intent tool for the Agentic RAG (best on BANKING77 with acceptable latency) and save its choice.